# 12 · Refresh and verify an accelerated dataset

Publish a managed Parquet snapshot, query its alias and prove reads no longer require the source file.

Open this notebook in Google Colab with a CPU runtime, or use a Linux x86-64
Jupyter environment with Python 3.11 or newer. No database account is required.
Run the cells in order. The first cell downloads a verified Kelvo preview binary
and installs pinned Python readers; it does not compile Kelvo.

Every analytical SQL query below runs through the **Kelvo executable**. PyArrow
reads its Arrow output, and pandas/matplotlib format small results. The pinned
DuckDB driver materializes execution before delivering Arrow batches; delivery
backpressure is not a worker memory bound.


**Data:** [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/),
[CC0 license](https://github.com/allisonhorst/palmerpenguins/blob/8957207b78d6ccd1b4654a9dd9c9041b657478ab/DESCRIPTION).
Citation: Horst, Hill and Gorman (2020), *palmerpenguins: Palmer Archipelago
(Antarctica) penguin data*, [DOI: 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218).
The helper downloads the pinned original CSV and verifies its SHA-256.


[All notebooks](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/README.md)
| [Kelvo documentation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/README.md)

## Set up a private lab


In [ ]:
# Default setup installs pinned Python packages and verifies the reviewed helper.
# Operators validating a checkout can explicitly set KELVO_NOTEBOOK_SKIP_INSTALL=1
# and KELVO_NOTEBOOK_HELPER=/absolute/path/to/kelvo_notebooks.py before starting.
import hashlib
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request

KELVO_NOTEBOOK_REF = "eacb7e2aedd8020575ecfede53db819112c40199"
KELVO_HELPER_SHA256 = "86696a66a3c903eb8f18d529fcef055e8bee72234da324b8776529138436d9a9"
if os.environ.get("KELVO_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "pyarrow==25.0.1", "pandas==2.3.3", "matplotlib==3.10.7", "PyYAML==6.0.3",
    ], check=True)

helper_directory = None
local_helper = os.environ.get("KELVO_NOTEBOOK_HELPER")
if local_helper:
    helper_path = Path(local_helper).expanduser().resolve(strict=True)
else:
    helper_url = (
        "https://raw.githubusercontent.com/SYNEHQ/kelvo-go/"
        + KELVO_NOTEBOOK_REF + "/notebooks/kelvo_notebooks.py"
    )
    request = urllib.request.Request(helper_url, headers={"User-Agent": "kelvo-notebooks"})
    with urllib.request.urlopen(request, timeout=60) as response:
        final_url = response.geturl()
        if urllib.parse.urlsplit(final_url).scheme != "https" or final_url != helper_url:
            raise ValueError("Helper download did not retain its pinned HTTPS URL")
        payload = response.read(1024 * 1024 + 1)
    if len(payload) > 1024 * 1024:
        raise ValueError("Unexpected helper size")
    if hashlib.sha256(payload).hexdigest() != KELVO_HELPER_SHA256:
        raise ValueError("Helper checksum mismatch")
    helper_directory = Path(tempfile.mkdtemp(prefix="kelvo-notebook-helper-"))
    helper_path = helper_directory / "kelvo_notebooks.py"
    helper_path.write_bytes(payload)

spec = importlib.util.spec_from_file_location("kelvo_notebooks", helper_path)
helper = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helper
spec.loader.exec_module(helper)
from kelvo_notebooks import Lab, remote_query

lab = Lab.create()
print("Kelvo workspace:", lab.root)


In [ ]:
import yaml
source = lab.root / "penguins-source.csv"
shutil.copyfile(lab.penguins(), source)
config = lab.catalog(
    sources=[{"id": "penguins", "type": "csv", "path": str(source)}],
    acceleration={
        "tenant_id": "notebook",
        "directory": str(lab.root / "snapshots"),
        "datasets": [{
            "id": "penguins_fast",
            "query": {"mode": "federated", "sources": ["penguins"], "sql": """
                SELECT species, island,
                       TRY_CAST(NULLIF(body_mass_g::VARCHAR, 'NA') AS INTEGER) AS body_mass_g
                FROM penguins
            """},
            "authorization_version": "public-data-v1",
            "refresh_interval": "0s", "max_age": "1h",
            "limits": {"max_rows": 10000, "max_bytes": 16777216,
                       "timeout": "60s", "memory_mb": 256, "threads": 2, "max_temp_mb": 256},
        }],
    },
)
print(config.read_text())


## Publish one complete generation

Refresh executes the configured source query. Status reports readiness; verify
checks the pinned generation's content. A manual refresh does not install a
background scheduler.


In [ ]:
refresh = yaml.safe_load(lab.command([
    "accelerate", "refresh", "--config", str(config), "--dataset", "penguins_fast"
]).stdout)
status = yaml.safe_load(lab.command([
    "accelerate", "status", "--config", str(config), "--dataset", "penguins_fast"
]).stdout)
verified = yaml.safe_load(lab.command([
    "accelerate", "verify", "--config", str(config), "--dataset", "penguins_fast"
]).stdout)
assert refresh["ready"] and status["ready"] and verified["ready"]
generation = refresh["snapshot"]["generation"]
assert generation == status["snapshot"]["generation"] == verified["snapshot"]["generation"]
print("Verified generation:", generation)


In [ ]:
sql = """
    SELECT species, COUNT(*)::BIGINT AS observations,
           COUNT(body_mass_g)::BIGINT AS measured
    FROM penguins_fast GROUP BY species ORDER BY species
"""
baseline = lab.query(config, sql, sources=["penguins_fast"])
assert sum(baseline.table()["observations"].to_pylist()) == 344
assert sum(baseline.table()["measured"].to_pylist()) == 342
assert baseline.stats["accelerations"][0]["generation"] == generation
display(baseline.table().to_pandas())


## Query with the extraction source unavailable

Only this notebook's private copy is renamed. The shared original download is
untouched. `finally` restores the copy even if the assertion fails.


In [ ]:
unavailable = source.with_suffix(".offline")
source.rename(unavailable)
try:
    independent = lab.query(config, sql, sources=["penguins_fast"])
    assert independent.table().equals(baseline.table(), check_metadata=True)
    assert independent.stats["accelerations"][0]["generation"] == generation
finally:
    unavailable.rename(source)
print("Snapshot query succeeded with its source file unavailable.")


This demonstrates source-independent reads, not a speed advantage over a
well-tuned database. Full refresh still scans and copies the complete configured
result. Incremental refresh/CDC and a global transaction across datasets are
separate capabilities. See [acceleration](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/acceleration.md).


## Clean up

Run this cell after inspecting or saving your results. Re-run the setup and data
cells if you want to restart the lesson after cleanup.


In [ ]:
# Close local processes and delete this lesson's private temporary workspace.
# Downloaded release/data caches, if configured by the helper, remain reusable.
lab.close()
if helper_directory is not None:
    shutil.rmtree(helper_directory, ignore_errors=True)
